# Microsoft Foundry · Agente analista de dados com Code Interpreter

Laboratório do curso de **Microsoft Foundry** (Seção 4 · Aula 15): um agente que recebe um **arquivo CSV**, escreve e **executa código Python** em uma sandbox e devolve um **gráfico** pronto, tudo a partir de um pedido em linguagem natural.

## Sumário

0. [Visão geral do projeto](#0.-Visão-geral-do-projeto)
1. [Instalação das bibliotecas](#1.-Instalação-das-bibliotecas)
2. [Variáveis de ambiente](#2.-Variáveis-de-ambiente)
3. [Conectando ao projeto do Foundry](#3.-Conectando-ao-projeto-do-Foundry)
4. [Cliente OpenAI do projeto](#4.-Cliente-OpenAI-do-projeto)
5. [Enviando o CSV ao Foundry](#5.-Enviando-o-CSV-ao-Foundry)
6. [Criando o agente com Code Interpreter](#6.-Criando-o-agente-com-Code-Interpreter)
7. [Criando a conversa](#7.-Criando-a-conversa)
8. [Pedindo o gráfico ao agente](#8.-Pedindo-o-gráfico-ao-agente)
9. [Baixando o gráfico gerado](#9.-Baixando-o-gráfico-gerado)
10. [Verificando no portal do Foundry](#10.-Verificando-no-portal-do-Foundry)
11. [Resumo dos aprendizados](#11.-Resumo-dos-aprendizados)
12. [Limpeza e custos](#12.-Limpeza-e-custos)
13. [Interface com Streamlit](#13.-Interface-com-Streamlit)

---
## 0. Visão geral do projeto

### 0.1 Problema

Modelos de linguagem são ótimos com texto, mas ruins em **contas exatas** e não conseguem "desenhar" gráficos. Analisar uma planilha normalmente exige alguém que saiba programar (pandas, matplotlib…).

### 0.2 Objetivo

- Enviar o arquivo `electronics_products.csv` ao projeto do **Microsoft Foundry**.
- Criar um agente com a ferramenta **Code Interpreter**, com acesso a esse arquivo.
- Pedir em português/inglês comum: *"crie um gráfico de colunas com produtos e preços"*.
- O agente **escreve e executa Python** em um ambiente isolado (sandbox) e devolve o **PNG** do gráfico, que baixamos pelo código.

### 0.3 Ferramentas e serviços da Azure

| Serviço / ferramenta | Papel no projeto |
| --- | --- |
| **Microsoft Foundry** (projeto) | Guarda o arquivo enviado (seção **Data**), o agente, as conversas e os traces. |
| **Foundry Agent Service** | Hospeda o `code-interpreter-agent` e orquestra a ferramenta. |
| **Code Interpreter** (`CodeInterpreterTool`) | Sandbox gerenciada que executa o Python escrito pelo agente (pandas, matplotlib…). |
| **Files API** (`openai_client.files`) | Upload do CSV para o projeto (`purpose="assistants"`). |
| **Containers API** (`openai_client.containers`) | Download dos arquivos gerados dentro da sandbox. |
| **Azure OpenAI** (deployment GPT-4) | Entende o pedido e escreve o código. |
| **Microsoft Entra ID** | Autenticação sem chave (`DefaultAzureCredential`). |

### 0.4 Os dados

O `electronics_products.csv` tem **50 produtos** com 4 colunas:

| productID | productName | price | colour |
| --- | --- | --- | --- |
| P001 | Printer | 172.15 | Green |
| P002 | Motherboard | 249.52 | White |
| P006 | Desktop | 1887.18 | Red |
| … | … | … | … |

### 0.5 Como funciona (visão simples)

![Fluxo do agente com Code Interpreter](fluxo_code_interpreter.svg)

### 0.6 Arquitetura

![Arquitetura do projeto](arquitetura.svg)

> 💡 **Para um leigo:** é como ter um analista de dados que, ao receber a planilha e o pedido, abre um computador isolado, escreve um pequeno programa, roda e te entrega o gráfico pronto.

---
## 1. Instalação das bibliotecas

| Biblioteca | Para que serve |
| --- | --- |
| `azure-ai-projects==2.0.0b2` | **Foundry SDK v2**: cliente do projeto (`AIProjectClient`), agentes e ferramentas. Versão *beta*, fixada para o código funcionar igual ao da aula. |
| `openai==1.109.1` | Cliente OpenAI, usado para criar conversas e enviar mensagens ao agente (Responses API). |
| `python-dotenv` | Lê o arquivo `.env` com as configurações. |
| `azure-identity` | Autenticação na Azure sem chave no código (`DefaultAzureCredential`). |


> ⚠️ Versões fixadas (`==`) evitam surpresas: SDKs em *beta* mudam nomes de classes e métodos entre versões.

In [ ]:
# Instalando as bibliotecas no mesmo ambiente do kernel do notebook (%pip)
# azure-ai-projects = Foundry SDK | openai = conversa com o agente
# python-dotenv = lê o .env | azure-identity = login na Azure sem chave no código
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

---
## 2. Variáveis de ambiente

As configurações ficam no arquivo **`.env`** (fora do código):

| Variável | O que é | Onde encontrar |
| --- | --- | --- |
| `FOUNDRY_PROJECT_ENDPOINT` | Endereço (URL) do projeto Foundry. | Portal do Foundry → página **Início** (*Overview*) → *Project endpoint*. |
| `MODEL_DEPLOYMENT_NAME` | Nome do deployment do modelo de chat. | Portal do Foundry → **Modelos + endpoints**. |


```
FOUNDRY_PROJECT_ENDPOINT=https://<seu-recurso>.services.ai.azure.com/api/projects/<seu-projeto>
MODEL_DEPLOYMENT_NAME=<nome-do-deployment>

```

> 🔒 **Segurança:** o `.env` guarda apenas **endereços e nomes** (nenhuma chave), porque a autenticação é feita pelo Microsoft Entra ID (seção seguinte). Se algum dia ele passar a ter segredos (chaves de API, tokens), não deve ser publicado: nesse caso, adicione `.env` ao `.gitignore`.

Classes importadas para este lab:

| Classe | Para que serve |
| --- | --- |
| `PromptAgentDefinition` | Define o agente: modelo + instruções + ferramentas. |
| `CodeInterpreterTool` | A ferramenta que executa Python. |
| `CodeInterpreterToolAuto` | Configura o **container** (sandbox) automático e quais arquivos ele enxerga (`file_ids`). |

In [ ]:
# Importando as bibliotecas e lendo as configurações do arquivo .env
# os.getenv busca cada valor pelo nome; se a variável não existir, retorna None
# CodeInterpreterTool / CodeInterpreterToolAuto = ferramenta e sandbox do Code Interpreter
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition, CodeInterpreterTool, CodeInterpreterToolAuto

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")

---
## 3. Conectando ao projeto do Foundry

O **`AIProjectClient`** é a porta de entrada do Python para o projeto no Foundry: por ele criamos agentes, listamos conexões e obtemos o cliente OpenAI.

- **`endpoint`**: o endereço do projeto (do `.env`).
- **`credential`**: o **`DefaultAzureCredential`** reaproveita um login que já existe na máquina (Azure CLI `az login`, conta Azure do VS Code, *Managed Identity* quando roda na Azure…). Assim, **nenhuma chave fica escrita no código**: quem autentica é o **Microsoft Entra ID**.

> 💡 Para rodar localmente, basta executar `az login` antes. O usuário precisa ter permissão no projeto (por exemplo, a função **Azure AI User**).

In [ ]:
# Criando o cliente do projeto Foundry: endpoint do .env + login da máquina (Entra ID)
# A conexão é "preguiçosa": erros de login só aparecem na primeira chamada real
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

---
## 4. Cliente OpenAI do projeto

O agente roda sobre um modelo **GPT** (Azure OpenAI). Para conversar com ele usamos a **Responses API** do SDK `openai`, e o próprio projeto entrega esse cliente já configurado com o endpoint e a autenticação do Foundry: `project_client.get_openai_client()`.

In [ ]:
# Obtendo o cliente OpenAI já autenticado no projeto Foundry
# É ele que cria conversas e envia mensagens ao agente (Responses API)
openai_client = project_client.get_openai_client()

---
## 5. Enviando o CSV ao Foundry

Para o agente usar um arquivo, ele primeiro precisa ser **registrado no projeto Foundry** (e não diretamente no agente):

- `openai_client.files.create(...)` envia o arquivo;
- `purpose="assistants"` indica que ele será usado por agentes;
- o retorno traz um **`file.id`**, que entregaremos ao agente na próxima seção.

> 💡 No portal, o arquivo aparece na seção **Data** (logo abaixo de Agentes). O mesmo vale para arquivos usados pela ferramenta *File Search*.

In [ ]:
# Enviando o CSV ao projeto Foundry; o retorno traz o file.id
# purpose="assistants" = arquivo para uso de agentes
# Upload the CSV file for the code interpreter to use
file = openai_client.files.create(purpose="assistants", file=open("./electronics_products.csv", "rb"))
print(f"File uploaded (id: {file.id})")

---
## 6. Criando o agente com Code Interpreter

| Campo | Valor | Significado |
| --- | --- | --- |
| `agent_name` | `code-interpreter-agent` | Nome do agente. |
| `instructions` | `"You are a helpful AI Assistant with code interpreter capabilities."` | *"Você é um assistente de IA útil com capacidade de interpretar código."* |
| `tools` | `CodeInterpreterTool(container=CodeInterpreterToolAuto(file_ids=[file.id]))` | Dá ao agente uma **sandbox Python** e, dentro dela, acesso ao CSV enviado. |

O `CodeInterpreterToolAuto` cria o container automaticamente: sempre que o agente usar o Code Interpreter, o arquivo com aquele `file_id` estará disponível para leitura.

In [ ]:
# Criando o agente com o Code Interpreter e acesso ao CSV (file_ids)
# Cada execução gera uma nova versão do agente
agent_name = "code-interpreter-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are a helpful AI Assistant with code interpreter capabilities.",
        tools = [
            CodeInterpreterTool(
                container = CodeInterpreterToolAuto(
                    file_ids = [file.id]
                )
            )
        ]
    ),
)

# printing the agent id
print(f"Agent created (id: {agent.id}, name: {agent.name}, version: {agent.version})")

---
## 7. Criando a conversa

A **conversation** é um objeto guardado na Azure que **armazena o histórico** entre o usuário e o agente, identificado pelo `conversation.id`. Todas as mensagens enviadas com o mesmo id fazem parte do mesmo diálogo, então o agente "lembra" o que já foi dito.

> 💡 No portal, o mesmo `conversation.id` aparece em **Traces**, com cada chamada de ferramenta e o consumo de tokens.

In [ ]:
# Criando a conversa que guarda o histórico (memória) entre usuário e agente
# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

---
## 8. Pedindo o gráfico ao agente

Pedido (em português): *"Você poderia criar um gráfico de colunas com os produtos no eixo x e seus respectivos preços no eixo y?"*

Aqui imprimimos apenas o **id da resposta**: o resultado mais importante não é o texto, e sim o **arquivo de imagem** que o agente gerou dentro da sandbox.

In [ ]:
# Enviando o pedido ao agente; ele vai escrever e executar Python na sandbox
# Guardamos a resposta inteira (response) para procurar o arquivo gerado
response = openai_client.responses.create(
    conversation = conversation.id,
    input = "Could you please create a column chart with products on the x-axis and their respective prices on the y-axis?",
    extra_body = {
        "agent": {
            "name": agent.name,
            "type": "agent_reference"
        }
    }
)

print(f"Response completed with id: {response.id}")

---
## 9. Baixando o gráfico gerado

O gráfico fica salvo **dentro do container** do Code Interpreter. A resposta informa onde ele está por meio de uma **anotação** na última mensagem:

| Caminho na resposta | O que é |
| --- | --- |
| `response.output[-1]` | Última saída (`ResponseOutputMessage`) |
| `.content[-1]` | Último conteúdo de texto (`ResponseOutputText`) |
| `.annotations[-1]` | Anotação do tipo **`container_file_citation`** |
| `.file_id`, `.filename`, `.container_id` | Os três dados necessários para baixar o arquivo |

Com eles, `openai_client.containers.files.content.retrieve(file_id=..., container_id=...)` devolve os bytes, que gravamos em um arquivo `.png` nesta pasta.

In [ ]:
# Procurando a anotação container_file_citation na resposta e baixando o PNG gerado
# file_id + container_id identificam o arquivo dentro da sandbox
file_id = ""
filename = ""
container_id = ""

# Get the last message which should contain file citations
last_message = response.output[-1]  # ResponseOutputMessage
if last_message.type == "message":
        # Get the last content item (contains the file annotations)
        text_content = last_message.content[-1]  # ResponseOutputText
        if text_content.type == "output_text":
            # Get the last annotation (most recent file)
            if text_content.annotations:
                file_citation = text_content.annotations[-1]  # AnnotationContainerFileCitation
                if file_citation.type == "container_file_citation":
                    file_id = file_citation.file_id
                    filename = file_citation.filename
                    container_id = file_citation.container_id
                    print(f"Found generated file: {filename} (ID: {file_id})")

# Download the generated file if available
if file_id and filename:
        file_content = openai_client.containers.files.content.retrieve(file_id=file_id, container_id=container_id)
        with open(filename, "wb") as f:
            f.write(file_content.read())
            print(f"File {filename} downloaded successfully.")
        print(f"File ready for download: {filename}")
else:
        print("No file generated in response")

✅ Na demonstração da aula, o arquivo baixado é um **gráfico de colunas** com os produtos no eixo x e os preços no eixo y, exatamente como pedido.

---
## 10. Verificando no portal do Foundry

Em **Agentes → `code-interpreter-agent` → Traces**, abrindo o `conversation.id`, dá para ver **o código que o agente executou**:

| Etapa | O que o agente fez |
| --- | --- |
| 1ª `code_interpreter_call` | `import pandas as pd`, leu o CSV e mostrou as primeiras linhas para entender a estrutura. |
| Mensagem intermediária | "O conjunto de dados contém as colunas relevantes: nome do produto e preço… vou criar um gráfico de colunas." O agente **conversa consigo mesmo** dentro da conversa. |
| 2ª `code_interpreter_call` | Código final com **matplotlib**, que gera e salva o gráfico. |
| Mensagem final | "Aqui está o gráfico de colunas…" + a anotação com `container_id`, `file_id` e `filename`. |

Na seção **Data** aparecem os arquivos enviados ao projeto (o CSV deste lab e, por exemplo, PDFs usados em labs de File Search).

---
## 11. Resumo dos aprendizados

- Arquivos usados por agentes são enviados **ao projeto** com `files.create(purpose="assistants")`, que devolve um `file_id`.
- O **Code Interpreter** dá ao agente uma **sandbox Python**; o `CodeInterpreterToolAuto(file_ids=[...])` coloca os arquivos dentro dela.
- O agente **escreve, executa e corrige** o próprio código (explora os dados e depois plota).
- Arquivos gerados voltam como anotação **`container_file_citation`** e são baixados com `containers.files.content.retrieve`.
- O **tracing** mostra cada trecho de código executado: a análise é **auditável**.

---
## 12. Limpeza e custos

| Recurso | Gera custo? |
| --- | --- |
| Chamadas ao modelo (tokens) | Sim, **por uso**: só quando você conversa com o agente. Ferramentas que trazem muito texto (páginas, APIs, documentos) aumentam bastante os tokens. |
| Code Interpreter | Sim, **por sessão** de container (cobrança por sessão ativa; confira a página de preços do Foundry). |
| Arquivos enviados (Data) | Armazenamento pequeno; apague em **Data** os arquivos que não usar mais. |

Ao terminar os estudos:

- **Portal do Foundry → Agentes**: apagar as versões do agente que não serão mais usadas.
- Ao final do curso, apagar o **grupo de recursos** no `portal.azure.com` remove tudo de uma vez.

> 💡 Vale criar um **orçamento com alerta** em *Cost Management → Budgets* para ser avisado de qualquer gasto inesperado.

---
## 13. Interface com Streamlit

A célula abaixo gera o arquivo **`app.py`**: o mesmo agente, agora com uma interface de chat no navegador. Ela reaproveita o código do notebook:

| Função do `app.py` | Código do notebook reaproveitado |
| --- | --- |
| `criar_clientes()` | `AIProjectClient` (seção 3) e `get_openai_client()` (seção 4) |
| `preparar_agente()` | `files.create` (seção 5) + `create_version` com `CodeInterpreterTool` (seção 6) |
| `nova_conversa() / perguntar()` | `conversations.create` (seção 7) e `responses.create` (seção 8) |
| `baixar_arquivos()` | a mesma lógica de anotações `container_file_citation` da seção 9 |

Pontos importantes:

- O Streamlit **reexecuta o script inteiro** a cada interação. Por isso os clientes ficam em cache (`@st.cache_resource`) e o `conversation.id` e o histórico ficam no `st.session_state`.
- O agente **não é recriado** a cada mensagem: o app usa o agente já criado neste notebook (pelo nome, sempre na versão mais recente).
- O gráfico aparece **direto no chat** (com botão de download) e o **código Python executado** fica em um bloco expansível.
- Na barra lateral é possível enviar **outro CSV**: o app faz o upload e publica uma **nova versão** do agente com esse arquivo.

**Como executar:**

```bash
pip install streamlit
az login
streamlit run app.py
```

O app abre em `http://localhost:8501` e usa o mesmo arquivo `.env` desta pasta.

In [ ]:
%%writefile app.py

import os
from pathlib import Path

import streamlit as st
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import CodeInterpreterTool, CodeInterpreterToolAuto, PromptAgentDefinition
from azure.core.exceptions import ClientAuthenticationError, HttpResponseError
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from openai import APIError

# ------------------------------------------------------------------
# 1. Configurações (as mesmas do notebook)
# ------------------------------------------------------------------
load_dotenv()

FOUNDRY_PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("MODEL_DEPLOYMENT_NAME")
AGENT_NAME = "code-interpreter-agent"
CSV_PADRAO = Path(__file__).parent / "electronics_products.csv"

st.set_page_config(page_title="Code Interpreter Agent · Microsoft Foundry", page_icon="📊", layout="wide")


# ------------------------------------------------------------------
# 2. Conexão com o projeto do Foundry
# ------------------------------------------------------------------
@st.cache_resource
def criar_clientes():
    project_client = AIProjectClient(
        endpoint=FOUNDRY_PROJECT_ENDPOINT,
        credential=DefaultAzureCredential(),
    )
    return project_client, project_client.get_openai_client()


# ------------------------------------------------------------------
# 3. Novo arquivo: upload + nova versão do agente (seções 5 e 6 do notebook)
# ------------------------------------------------------------------
def preparar_agente(nome_arquivo, conteudo):
    project_client, openai_client = criar_clientes()
    # Upload do CSV para o projeto: devolve o file_id
    arquivo = openai_client.files.create(purpose="assistants", file=(nome_arquivo, conteudo))
    # Nova versão do agente com acesso a esse arquivo na sandbox
    agente = project_client.agents.create_version(
        agent_name=AGENT_NAME,
        definition=PromptAgentDefinition(
            model=MODEL_DEPLOYMENT_NAME,
            instructions="You are a helpful AI Assistant with code interpreter capabilities.",
            tools=[CodeInterpreterTool(container=CodeInterpreterToolAuto(file_ids=[arquivo.id]))],
        ),
    )
    return agente.version


# ------------------------------------------------------------------
# 4. Conversa e pergunta
# ------------------------------------------------------------------
def nova_conversa():
    st.session_state.conversation_id = criar_clientes()[1].conversations.create().id
    st.session_state.messages = []


def perguntar(texto):
    return criar_clientes()[1].responses.create(
        conversation=st.session_state.conversation_id,
        extra_body={"agent": {"name": AGENT_NAME, "type": "agent_reference"}},
        input=texto,
    )


# ------------------------------------------------------------------
# 5. O que o agente produziu: código executado e arquivos gerados
# ------------------------------------------------------------------
def extrair_codigos(response):
    # Cada code_interpreter_call guarda o código Python que rodou na sandbox
    return [item.code for item in response.output if item.type == "code_interpreter_call" and getattr(item, "code", None)]


def baixar_arquivos(response):
    # Mesma lógica da seção 9: procura as anotações container_file_citation e baixa cada arquivo
    _, openai_client = criar_clientes()
    arquivos = []
    for item in response.output:
        if item.type != "message":
            continue
        for conteudo in item.content:
            for anotacao in getattr(conteudo, "annotations", None) or []:
                if anotacao.type == "container_file_citation":
                    dados = openai_client.containers.files.content.retrieve(
                        file_id=anotacao.file_id, container_id=anotacao.container_id
                    ).read()
                    arquivos.append({"nome": anotacao.filename, "dados": dados})
    return arquivos


def mostrar_extras(codigos, arquivos, chave):
    for arquivo in arquivos or []:
        if arquivo["nome"].lower().endswith((".png", ".jpg", ".jpeg")):
            st.image(arquivo["dados"], caption=arquivo["nome"])
        st.download_button(f"⬇️ {arquivo['nome']}", arquivo["dados"], file_name=arquivo["nome"], key=f"{chave}-{arquivo['nome']}")
    if codigos:
        with st.expander(f"🐍 Código executado na sandbox ({len(codigos)} execuções)"):
            for codigo in codigos:
                st.code(codigo, language="python")


# ------------------------------------------------------------------
# 6. Interface
# ------------------------------------------------------------------
st.title("📊 Agente analista de dados")
st.caption("Foundry Agent Service + Code Interpreter: o agente escreve e executa Python para analisar o seu CSV.")

if not FOUNDRY_PROJECT_ENDPOINT or not MODEL_DEPLOYMENT_NAME:
    st.error("Configure FOUNDRY_PROJECT_ENDPOINT e MODEL_DEPLOYMENT_NAME no arquivo `.env`.")
    st.stop()

if "conversation_id" not in st.session_state:
    try:
        nova_conversa()
    except ClientAuthenticationError:
        st.error("Falha no login da Azure. Execute `az login` no terminal e recarregue a página.")
        st.stop()
    except (HttpResponseError, APIError) as erro:
        st.error(f"Erro ao criar a conversa no Foundry: {erro}")
        st.stop()

with st.sidebar:
    st.header("📁 Dados")
    st.caption(f"Por padrão o agente usa o CSV enviado no notebook (`{CSV_PADRAO.name}`).")
    novo_csv = st.file_uploader("Analisar outro CSV", type="csv")
    if novo_csv and st.button("🚀 Enviar e publicar nova versão"):
        try:
            versao = preparar_agente(novo_csv.name, novo_csv.getvalue())
            nova_conversa()
            st.success(f"Agente versão {versao} pronto para `{novo_csv.name}`.")
        except (HttpResponseError, ClientAuthenticationError, APIError) as erro:
            st.error(f"Erro ao preparar o agente: {erro}")
    st.divider()
    st.caption(f"Agente: `{AGENT_NAME}`  \nConversa: `{st.session_state.conversation_id}`")
    if st.button("🗑️ Nova conversa"):
        nova_conversa()
        st.rerun()

for i, message in enumerate(st.session_state.messages):
    with st.chat_message(message["role"]):
        st.markdown(message["content"])
        mostrar_extras(message.get("codigos"), message.get("arquivos"), chave=i)

if prompt := st.chat_input("Ex.: crie um gráfico de colunas com produtos no eixo x e preços no eixo y"):
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Escrevendo e executando o código na sandbox..."):
            try:
                response = perguntar(prompt)
                codigos, arquivos = extrair_codigos(response), baixar_arquivos(response)
            except ClientAuthenticationError:
                st.error("Falha no login da Azure. Execute `az login` no terminal.")
            except (HttpResponseError, APIError) as erro:
                st.error(f"Erro ao consultar o agente: {erro}")
            else:
                st.markdown(response.output_text)
                mostrar_extras(codigos, arquivos, chave="nova")
                st.session_state.messages.append({"role": "assistant", "content": response.output_text,
                                                  "codigos": codigos, "arquivos": arquivos})
